# **[DEPRECATED]**

# Submission pipe: answer.csv

In [1]:
import sys

sys.path.append("..")

from collections import defaultdict

import bm25s
import numpy as np
import pandas as pd

from eval import DEFAULT_SEARCH_COLS, make_val_split, recall_at_50
from pipeline.geo import build_geo_tree, build_loc_centroid, haversine
from preprocessing import clean_text, extract_label_value, tokenize

train = pd.read_parquet("../data/train.parquet")
bench_items = pd.read_parquet("../data/benchmark_items.parquet")
bench_queries = pd.read_parquet("../data/benchmark_queries.parquet")
print("train:", len(train), "bench_items:", len(bench_items), "bench_queries:", len(bench_queries))

train: 497673 bench_items: 189212 bench_queries: 2452


## BM25-индекс на benchmark_items

In [2]:
bench_items = bench_items.copy()
bench_items["item_description_raw"] = bench_items["item_description_raw"].fillna("")
bench_items["text"] = (
    bench_items["item_title_raw"] + " "
    + bench_items["item_description_raw"] + " "
    + bench_items["item_infm_params_text"]
)
corpus_item_ids_bench = bench_items["item_id"].tolist()

BM25_BENCH_CACHE = "../data/bm25s_index_bench"
import os
if os.path.exists(BM25_BENCH_CACHE):
    retriever_bench = bm25s.BM25.load(BM25_BENCH_CACHE, load_corpus=False)
    print("BM25 (bench) загружен из кэша")
else:
    bench_items["tokens"] = bench_items["text"].apply(tokenize)
    retriever_bench = bm25s.BM25()
    retriever_bench.index(bench_items["tokens"].tolist(), show_progress=True)
    retriever_bench.save(BM25_BENCH_CACHE)
    print("BM25 (bench) посчитан и закэширован")

BM25 (bench) загружен из кэша


## Статистики на всём train (без holdout - это финальный прогон)

In [3]:
def build_lookup(item_ids, keys):
    lookup = defaultdict(dict)
    for key, item_id in zip(keys, item_ids):
        d = lookup[key]
        d[item_id] = d.get(item_id, 0) + 1
    return lookup


context_train = train[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1)
query_norm_train = train["search_query"].str.lower().str.strip()
context_lookup = build_lookup(train["item_id"], context_train)
text_lookup = build_lookup(train["item_id"], query_norm_train)
item_popularity = train["item_id"].value_counts()
print("lookup-таблицы построены на", len(train), "строках train")

lookup-таблицы построены на 497673 строках train


## Гео: центроид локации из train, поиск по дистанции внутри benchmark_items

In [4]:
loc_centroid = build_loc_centroid(train)
geo_tree, geo_item_ids = build_geo_tree(
    bench_items[["item_id", "item_location_id", "item_latitude", "item_longitude"]]
)
print("центроидов:", len(loc_centroid), "| точек в гео-дереве (benchmark_items):", len(geo_item_ids))

coverage = bench_queries["search_location_id"].isin(loc_centroid.index).mean()
print("доля benchmark_queries с центроидом:", coverage)

центроидов: 2637 | точек в гео-дереве (benchmark_items): 189211
доля benchmark_queries с центроидом: 0.8266721044045677


## Фильтр-канал и гео exact-match+популярность (на benchmark_items, популярность из train - но реальный overlap мал, item из benchmark_items чаще всего НЕ встречался в train вообще)

In [5]:
item_vid_series = bench_items.set_index("item_id")["item_infm_params_text"].apply(
    lambda t: extract_label_value(t, "Вид услуги")
)
item_tip_series = bench_items.set_index("item_id")["item_infm_params_text"].apply(
    lambda t: extract_label_value(t, "Тип услуги")
)


def build_value_lookup(value_series, popularity):
    lookup = defaultdict(list)
    for item_id, value in value_series.items():
        if value:
            lookup[value].append(item_id)
    for value in lookup:
        lookup[value].sort(key=lambda iid: -popularity.get(iid, 0))
    return lookup


vid_lookup = build_value_lookup(item_vid_series, item_popularity)
tip_lookup = build_value_lookup(item_tip_series, item_popularity)

location_to_items = defaultdict(list)
for iid, loc in bench_items.set_index("item_id")["item_location_id"].items():
    location_to_items[loc].append(iid)
for loc in location_to_items:
    location_to_items[loc].sort(key=lambda iid: -item_popularity.get(iid, 0))

n_overlap = bench_items["item_id"].isin(item_popularity.index).sum()
print(f"объявлений benchmark_items с ненулевой train-популярностью: {n_overlap} ({n_overlap / len(bench_items) * 100:.2f}%)")

объявлений benchmark_items с ненулевой train-популярностью: 18142 (9.59%)


## Переобучение CatBoost на всём train (без holdout - это финальный прогон)

Обучающие пары строятся в корпусе **train** (не benchmark_items - тот появится только на этапе применения модели к `bench_queries`). Инфра каналов (BM25/dense/geo на train-каталоге) для этого нужна отдельно от той, что уже построена на `benchmark_items` выше.

In [6]:
items_corpus_train = train.drop_duplicates("item_id")[
    ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text"]
].reset_index(drop=True)
corpus_item_ids_bm25_train = items_corpus_train["item_id"].tolist()
retriever_train = bm25s.BM25.load("../data/bm25s_index", load_corpus=False)
print("BM25 (train) загружен")

BM25 (train) загружен


In [7]:
import faiss

corpus_embeddings_train = np.load("../data/embeddings/e5_base_enriched/512.npy")
dense_item_ids_train = np.load("../data/embeddings/e5_base_enriched/512_item_ids.npy", allow_pickle=True).tolist()
print("dense-эмбеды (train) загружены:", corpus_embeddings_train.shape)

dense-эмбеды (train) загружены: (344825, 768)


In [8]:
index_train = faiss.IndexFlatIP(corpus_embeddings_train.shape[1])
index_train.add(corpus_embeddings_train)
del corpus_embeddings_train
print("faiss-индекс (train) построен")

faiss-индекс (train) построен


In [9]:
import torch
from sentence_transformers import SentenceTransformer

device = "mps" if torch.backends.mps.is_available() else "cpu"
embed_model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)
embed_model.max_seq_length = 512
print("модель e5-base загружена, device:", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

модель e5-base загружена, device: mps


In [10]:
geo_tree_train, geo_item_ids_train = build_geo_tree(
    train.drop_duplicates("item_id")[["item_id", "item_location_id", "item_latitude", "item_longitude"]]
)
print("гео-дерево (train) построено:", len(geo_item_ids_train))

гео-дерево (train) построено: 344822


In [11]:
item_meta_train = train.drop_duplicates("item_id").set_index("item_id")[[
    "item_location_id", "item_latitude", "item_longitude",
    "item_price", "item_rating", "item_rating_reviews_count",
    "item_is_phone_hidden", "item_is_message_forbidden",
    "item_infm_params_text",
]].copy()
item_meta_train["item_latitude"] = item_meta_train["item_latitude"].astype(float)
item_meta_train["item_longitude"] = item_meta_train["item_longitude"].astype(float)
item_meta_train["item_price"] = item_meta_train["item_price"].astype(float).replace(-1, np.nan)
print("item_meta (train) построен:", item_meta_train.shape)

item_meta (train) построен: (344825, 9)


In [12]:
title_words_train = {
    iid: set(t.lower().split())
    for iid, t in zip(items_corpus_train["item_id"], items_corpus_train["item_title_raw"])
}
print("title_words (train) построен:", len(title_words_train))

title_words (train) построен: 344825


In [13]:
item_vid_series_train = item_meta_train["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
item_tip_series_train = item_meta_train["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))
print("Вид/Тип услуги (train) распарсены")

Вид/Тип услуги (train) распарсены


In [14]:
def build_stat_tables_train(stat_part):
    def _build_lookup(item_ids, keys):
        lookup = defaultdict(dict)
        for key, item_id in zip(keys, item_ids):
            d = lookup[key]
            d[item_id] = d.get(item_id, 0) + 1
        return lookup

    context_p = stat_part[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1)
    query_norm_p = stat_part["search_query"].str.lower().str.strip()
    ctx_lookup = _build_lookup(stat_part["item_id"], context_p)
    txt_lookup = _build_lookup(stat_part["item_id"], query_norm_p)
    pop = stat_part["item_id"].value_counts()
    centroid = build_loc_centroid(stat_part)

    vid_lu = build_value_lookup(item_vid_series_train, pop)
    tip_lu = build_value_lookup(item_tip_series_train, pop)

    loc_to_items = defaultdict(list)
    for iid, loc in item_meta_train["item_location_id"].items():
        loc_to_items[loc].append(iid)
    for loc in loc_to_items:
        loc_to_items[loc].sort(key=lambda iid: -pop.get(iid, 0))

    return ctx_lookup, txt_lookup, pop, centroid, vid_lu, tip_lu, loc_to_items


def get_pool_and_channels_train(query_df, context_lookup, text_lookup, item_popularity, loc_centroid, vid_lookup, tip_lookup, location_to_items, k=100, batch=2000):
    all_pools, all_channels = {}, {}
    for start in range(0, len(query_df), batch):
        qdf = query_df.iloc[start:start + batch].reset_index(drop=True)
        n = len(qdf)
        val_tokens = qdf["search_query"].apply(tokenize).tolist()
        bm25_results, bm25_scores = retriever_train.retrieve(
            val_tokens, corpus=corpus_item_ids_bm25_train, k=k, show_progress=False, n_threads=0
        )
        query_texts = qdf["search_query"].apply(clean_text).apply(lambda t: f"query: {t}").tolist()
        q_emb = embed_model.encode(query_texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
        dense_scores_all, dense_idx_all = index_train.search(q_emb, k)

        query_coords = loc_centroid.reindex(qdf["search_location_id"])[["item_latitude", "item_longitude"]].to_numpy()
        valid_mask = ~np.isnan(query_coords).any(axis=1)
        geo_dist_all = np.full((n, k), np.nan)
        geo_idx_all = np.full((n, k), -1, dtype=int)
        if valid_mask.any():
            d, idx = geo_tree_train.query(np.radians(query_coords[valid_mask]), k=k)
            geo_dist_all[valid_mask] = d * 6371.0
            geo_idx_all[valid_mask] = idx

        contexts = qdf[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1).tolist()
        qnorms = qdf["search_query"].str.lower().str.strip().tolist()
        search_params_list = qdf["search_infm_params_text"].tolist()
        search_loc_list = qdf["search_location_id"].tolist()

        for i, cid in enumerate(qdf["context_id"]):
            bm25_ch = {bm25_results[i][r]: (r + 1, float(bm25_scores[i][r])) for r in range(k)}
            dense_ch = {dense_item_ids_train[dense_idx_all[i][r]]: (r + 1, float(dense_scores_all[i][r])) for r in range(k) if dense_idx_all[i][r] != -1}

            scores = {}
            for item_id, cnt in context_lookup.get(contexts[i], {}).items():
                scores[item_id] = scores.get(item_id, 0) + cnt * 2
            for item_id, cnt in text_lookup.get(qnorms[i], {}).items():
                scores[item_id] = scores.get(item_id, 0) + cnt
            ranked_lookup = sorted(scores.items(), key=lambda x: -x[1])[:k]
            lookup_ch = {iid: (r + 1, s) for r, (iid, s) in enumerate(ranked_lookup)}

            geo_ch = {}
            if geo_idx_all[i][0] != -1:
                for r in range(k):
                    gi = geo_idx_all[i][r]
                    if gi == -1:
                        continue
                    geo_ch[geo_item_ids_train[gi]] = (r + 1, float(geo_dist_all[i][r]))

            tip = extract_label_value(search_params_list[i], "Тип услуги")
            vid = extract_label_value(search_params_list[i], "Вид услуги")
            filter_cands = tip_lookup.get(tip, [])[:k] if tip else (vid_lookup.get(vid, [])[:k] if vid else [])
            filter_ch = {iid: (r + 1, float(item_popularity.get(iid, 0))) for r, iid in enumerate(filter_cands)}

            geo_exact_cands = location_to_items.get(search_loc_list[i], [])[:k]
            geo_exact_ch = {iid: (r + 1, float(item_popularity.get(iid, 0))) for r, iid in enumerate(geo_exact_cands)}

            pool = list(dict.fromkeys(
                list(bm25_ch.keys()) + list(dense_ch.keys()) + list(lookup_ch.keys()) + list(geo_ch.keys())
                + list(filter_ch.keys()) + list(geo_exact_ch.keys())
            ))
            all_pools[cid] = pool
            all_channels[cid] = {
                "bm25": bm25_ch, "dense": dense_ch, "lookup": lookup_ch, "geo": geo_ch,
                "filter": filter_ch, "geo_exact": geo_exact_ch,
            }
    return all_pools, all_channels


FEATURE_COLS = [
    "geo_distance_km", "geo_exact_match", "vid_uslugi_match", "tip_uslugi_match",
    "item_price", "item_rating", "item_rating_reviews_count",
    "item_is_phone_hidden", "item_is_message_forbidden", "item_popularity", "text_overlap",
    "bm25_rank", "bm25_present", "bm25_score",
    "dense_rank", "dense_present", "dense_score",
    "lookup_rank", "lookup_present", "lookup_score",
    "geo_rank", "geo_present", "geo_score",
    "filter_rank", "filter_present", "filter_score",
    "geo_exact_rank", "geo_exact_present", "geo_exact_score",
]


def assemble_features_train(query_df, pools, channels, pop, loc_centroid):
    rows = []
    query_rows = {row["context_id"]: row for _, row in query_df.iterrows()}
    for cid, pool in pools.items():
        if not pool:
            continue
        qrow = query_rows[cid]
        ch = channels[cid]
        meta = item_meta_train.reindex(pool)

        search_loc = qrow["search_location_id"]
        if search_loc in loc_centroid.index:
            c_lat, c_lon = loc_centroid.loc[search_loc, ["item_latitude", "item_longitude"]]
            geo_distance_km = haversine(c_lat, c_lon, meta["item_latitude"], meta["item_longitude"]).to_numpy()
        else:
            geo_distance_km = np.full(len(pool), np.nan)
        geo_exact_match = (meta["item_location_id"] == search_loc).astype(int).to_numpy()

        q_vid = extract_label_value(qrow["search_infm_params_text"], "Вид услуги")
        q_tip = extract_label_value(qrow["search_infm_params_text"], "Тип услуги")
        item_vid = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
        item_tip = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))

        q_words = set(qrow["search_query"].lower().split())
        text_overlap = np.array([len(q_words & title_words_train.get(iid, set())) for iid in pool])

        df = pd.DataFrame({
            "context_id": cid,
            "item_id": pool,
            "geo_distance_km": geo_distance_km,
            "geo_exact_match": geo_exact_match,
            "vid_uslugi_match": (item_vid == q_vid).astype(int).to_numpy() if q_vid else np.nan,
            "tip_uslugi_match": (item_tip == q_tip).astype(int).to_numpy() if q_tip else np.nan,
            "item_price": meta["item_price"].to_numpy(),
            "item_rating": meta["item_rating"].to_numpy(),
            "item_rating_reviews_count": meta["item_rating_reviews_count"].to_numpy(),
            "item_is_phone_hidden": meta["item_is_phone_hidden"].astype(int).to_numpy(),
            "item_is_message_forbidden": meta["item_is_message_forbidden"].astype(int).to_numpy(),
            "item_popularity": [pop.get(iid, 0) for iid in pool],
            "text_overlap": text_overlap,
        })
        for name, cmap in ch.items():
            ranks = np.array([cmap.get(iid, (0, np.nan))[0] for iid in pool])
            scores = np.array([cmap.get(iid, (0, np.nan))[1] for iid in pool])
            df[f"{name}_rank"] = ranks
            df[f"{name}_present"] = (ranks > 0).astype(int)
            df[f"{name}_score"] = scores
        rows.append(df)
    return pd.concat(rows, ignore_index=True)

In [15]:
fit_part, cb_train_queries, cb_train_relevant = make_val_split(train, val_size=0.05, random_state=7)

SAMPLE_N = 8000
cb_train_queries = cb_train_queries.sample(min(SAMPLE_N, len(cb_train_queries)), random_state=1).reset_index(drop=True)
print("fit_part:", len(fit_part), "cb_train_queries (сэмпл):", len(cb_train_queries))

cb_context_lookup, cb_text_lookup, cb_item_popularity, cb_loc_centroid, cb_vid_lookup, cb_tip_lookup, cb_location_to_items = build_stat_tables_train(fit_part)
pools_train, channels_train = get_pool_and_channels_train(
    cb_train_queries, cb_context_lookup, cb_text_lookup, cb_item_popularity, cb_loc_centroid, cb_vid_lookup, cb_tip_lookup, cb_location_to_items, k=100
)
print("средний размер пула (train):", np.mean([len(p) for p in pools_train.values()]))

feat_train = assemble_features_train(cb_train_queries, pools_train, channels_train, cb_item_popularity, cb_loc_centroid)
feat_train["label"] = feat_train.apply(
    lambda r: int(r["item_id"] in cb_train_relevant.get(r["context_id"], set())), axis=1
)
feat_train = feat_train.sort_values("context_id").reset_index(drop=True)
print("feat_train:", feat_train.shape, "позитивов:", feat_train["label"].sum())

fit_part: 472303 cb_train_queries (сэмпл): 8000


средний размер пула (train): 420.778625


feat_train: (3366229, 32) позитивов: 5231


In [16]:
from catboost import CatBoostRanker, Pool

X = feat_train[FEATURE_COLS]
y = feat_train["label"]
group_id = feat_train["context_id"]

uniq_ctx = feat_train["context_id"].unique()
rng = np.random.RandomState(0)
holdout_ctx = set(rng.choice(uniq_ctx, size=int(len(uniq_ctx) * 0.1), replace=False))
train_mask = ~feat_train["context_id"].isin(holdout_ctx)

train_pool = Pool(X[train_mask], y[train_mask], group_id=group_id[train_mask])
eval_pool = Pool(X[~train_mask], y[~train_mask], group_id=group_id[~train_mask])

catboost_model = CatBoostRanker(
    loss_function="PairLogit", iterations=300, learning_rate=0.1, depth=6,
    random_seed=42, verbose=50,
)
catboost_model.fit(train_pool, eval_set=eval_pool)

CATBOOST_MODEL_PATH = "../data/catboost_reranker_prod.cbm"
catboost_model.save_model(CATBOOST_MODEL_PATH)
print("модель сохранена:", CATBOOST_MODEL_PATH)

importances = catboost_model.get_feature_importance(train_pool)
pd.DataFrame({"feature": FEATURE_COLS, "importance": importances}).sort_values("importance", ascending=False)

0:	learn: 0.3839875	test: 0.3877098	best: 0.3877098 (0)	total: 604ms	remaining: 3m


50:	learn: 0.0321704	test: 0.0441493	best: 0.0441184 (44)	total: 26.8s	remaining: 2m 11s


100:	learn: 0.0264346	test: 0.0433578	best: 0.0421099 (73)	total: 52.1s	remaining: 1m 42s


150:	learn: 0.0223761	test: 0.0444231	best: 0.0421099 (73)	total: 1m 17s	remaining: 1m 16s


200:	learn: 0.0190522	test: 0.0453177	best: 0.0421099 (73)	total: 1m 42s	remaining: 50.4s


250:	learn: 0.0166913	test: 0.0498667	best: 0.0421099 (73)	total: 2m 7s	remaining: 24.9s


299:	learn: 0.0150116	test: 0.0524465	best: 0.0421099 (73)	total: 2m 32s	remaining: 0us

bestTest = 0.04210989227
bestIteration = 73

Shrink model to first 74 iterations.
модель сохранена: ../data/catboost_reranker_prod.cbm


,feature,importance
9,item_popularity,0.030078
0,geo_distance_km,0.025080
2,vid_uslugi_match,0.010541
10,text_overlap,0.007592
1,geo_exact_match,0.006968
13,bm25_score,0.005127
11,bm25_rank,0.000794
27,geo_exact_present,0.000720
3,tip_uslugi_match,0.000690
6,item_rating_reviews_count,0.000634


## Пул кандидатов для bench_queries на настоящем benchmark_items

In [17]:
import faiss

corpus_embeddings_bench = np.load("../data/embeddings/e5_base_bench/512.npy")
dense_item_ids_bench = np.load("../data/embeddings/e5_base_bench/512_item_ids.npy", allow_pickle=True).tolist()
index_bench = faiss.IndexFlatIP(corpus_embeddings_bench.shape[1])
index_bench.add(corpus_embeddings_bench)
del corpus_embeddings_bench
print("dense (bench) faiss-индекс построен:", len(dense_item_ids_bench))

dense (bench) faiss-индекс построен: 189212


In [18]:
bench_item_meta = bench_items.set_index("item_id")[[
    "item_location_id", "item_latitude", "item_longitude",
    "item_price", "item_rating", "item_rating_reviews_count",
    "item_is_phone_hidden", "item_is_message_forbidden",
    "item_infm_params_text",
]].copy()
bench_item_meta["item_latitude"] = bench_item_meta["item_latitude"].astype(float)
bench_item_meta["item_longitude"] = bench_item_meta["item_longitude"].astype(float)
bench_item_meta["item_price"] = bench_item_meta["item_price"].astype(float).replace(-1, np.nan)

bench_title_words = {
    iid: set(t.lower().split())
    for iid, t in zip(bench_items["item_id"], bench_items["item_title_raw"])
}
print("bench item_meta/title_words готовы:", bench_item_meta.shape)

bench item_meta/title_words готовы: (189212, 9)


In [19]:
def get_pool_and_channels_bench(query_df, k=100, batch=500):
    all_pools, all_channels = {}, {}
    for start in range(0, len(query_df), batch):
        qdf = query_df.iloc[start:start + batch].reset_index(drop=True)
        n = len(qdf)
        q_tokens = qdf["search_query"].apply(tokenize).tolist()
        bm25_results, bm25_scores = retriever_bench.retrieve(
            q_tokens, corpus=corpus_item_ids_bench, k=k, show_progress=False, n_threads=0
        )
        query_texts = qdf["search_query"].apply(clean_text).apply(lambda t: f"query: {t}").tolist()
        q_emb = embed_model.encode(query_texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
        dense_scores_all, dense_idx_all = index_bench.search(q_emb, k)

        query_coords = loc_centroid.reindex(qdf["search_location_id"])[["item_latitude", "item_longitude"]].to_numpy()
        valid_mask = ~np.isnan(query_coords).any(axis=1)
        geo_dist_all = np.full((n, k), np.nan)
        geo_idx_all = np.full((n, k), -1, dtype=int)
        if valid_mask.any():
            d, idx = geo_tree.query(np.radians(query_coords[valid_mask]), k=k)
            geo_dist_all[valid_mask] = d * 6371.0
            geo_idx_all[valid_mask] = idx

        contexts = qdf[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1).tolist()
        qnorms = qdf["search_query"].str.lower().str.strip().tolist()
        search_params_list = qdf["search_infm_params_text"].tolist()
        search_loc_list = qdf["search_location_id"].tolist()

        for i, cid in enumerate(qdf["query_id"]):
            bm25_ch = {bm25_results[i][r]: (r + 1, float(bm25_scores[i][r])) for r in range(k)}
            dense_ch = {dense_item_ids_bench[dense_idx_all[i][r]]: (r + 1, float(dense_scores_all[i][r])) for r in range(k) if dense_idx_all[i][r] != -1}

            scores = {}
            for item_id, cnt in context_lookup.get(contexts[i], {}).items():
                if item_id in bench_item_meta.index:
                    scores[item_id] = scores.get(item_id, 0) + cnt * 2
            for item_id, cnt in text_lookup.get(qnorms[i], {}).items():
                if item_id in bench_item_meta.index:
                    scores[item_id] = scores.get(item_id, 0) + cnt
            ranked_lookup = sorted(scores.items(), key=lambda x: -x[1])[:k]
            lookup_ch = {iid: (r + 1, s) for r, (iid, s) in enumerate(ranked_lookup)}

            geo_ch = {}
            if geo_idx_all[i][0] != -1:
                for r in range(k):
                    gi = geo_idx_all[i][r]
                    if gi == -1:
                        continue
                    geo_ch[geo_item_ids[gi]] = (r + 1, float(geo_dist_all[i][r]))

            tip = extract_label_value(search_params_list[i], "Тип услуги")
            vid = extract_label_value(search_params_list[i], "Вид услуги")
            filter_cands = tip_lookup.get(tip, [])[:k] if tip else (vid_lookup.get(vid, [])[:k] if vid else [])
            filter_ch = {iid: (r + 1, float(item_popularity.get(iid, 0))) for r, iid in enumerate(filter_cands)}

            geo_exact_cands = location_to_items.get(search_loc_list[i], [])[:k]
            geo_exact_ch = {iid: (r + 1, float(item_popularity.get(iid, 0))) for r, iid in enumerate(geo_exact_cands)}

            pool = list(dict.fromkeys(
                list(bm25_ch.keys()) + list(dense_ch.keys()) + list(lookup_ch.keys()) + list(geo_ch.keys())
                + list(filter_ch.keys()) + list(geo_exact_ch.keys())
            ))
            all_pools[cid] = pool
            all_channels[cid] = {
                "bm25": bm25_ch, "dense": dense_ch, "lookup": lookup_ch, "geo": geo_ch,
                "filter": filter_ch, "geo_exact": geo_exact_ch,
            }
    return all_pools, all_channels


pools_bench, channels_bench = get_pool_and_channels_bench(bench_queries, k=100)
print("средний размер пула (bench):", np.mean([len(p) for p in pools_bench.values()]))

средний размер пула (bench): 359.89233278955953


## Признаки + юзаем сохранённую модель CatBoost

In [20]:
def assemble_features_bench(query_df, pools, channels):
    rows = []
    query_rows = {row["query_id"]: row for _, row in query_df.iterrows()}
    for cid, pool in pools.items():
        if not pool:
            continue
        qrow = query_rows[cid]
        ch = channels[cid]
        meta = bench_item_meta.reindex(pool)

        search_loc = qrow["search_location_id"]
        if search_loc in loc_centroid.index:
            c_lat, c_lon = loc_centroid.loc[search_loc, ["item_latitude", "item_longitude"]]
            geo_distance_km = haversine(c_lat, c_lon, meta["item_latitude"], meta["item_longitude"]).to_numpy()
        else:
            geo_distance_km = np.full(len(pool), np.nan)
        geo_exact_match = (meta["item_location_id"] == search_loc).astype(int).to_numpy()

        q_vid = extract_label_value(qrow["search_infm_params_text"], "Вид услуги")
        q_tip = extract_label_value(qrow["search_infm_params_text"], "Тип услуги")
        item_vid = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
        item_tip = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))

        q_words = set(qrow["search_query"].lower().split())
        text_overlap = np.array([len(q_words & bench_title_words.get(iid, set())) for iid in pool])

        df = pd.DataFrame({
            "query_id": cid,
            "item_id": pool,
            "geo_distance_km": geo_distance_km,
            "geo_exact_match": geo_exact_match,
            "vid_uslugi_match": (item_vid == q_vid).astype(int).to_numpy() if q_vid else np.nan,
            "tip_uslugi_match": (item_tip == q_tip).astype(int).to_numpy() if q_tip else np.nan,
            "item_price": meta["item_price"].to_numpy(),
            "item_rating": meta["item_rating"].to_numpy(),
            "item_rating_reviews_count": meta["item_rating_reviews_count"].to_numpy(),
            "item_is_phone_hidden": meta["item_is_phone_hidden"].astype(int).to_numpy(),
            "item_is_message_forbidden": meta["item_is_message_forbidden"].astype(int).to_numpy(),
            "item_popularity": [item_popularity.get(iid, 0) for iid in pool],
            "text_overlap": text_overlap,
        })
        for name, cmap in ch.items():
            ranks = np.array([cmap.get(iid, (0, np.nan))[0] for iid in pool])
            scores = np.array([cmap.get(iid, (0, np.nan))[1] for iid in pool])
            df[f"{name}_rank"] = ranks
            df[f"{name}_present"] = (ranks > 0).astype(int)
            df[f"{name}_score"] = scores
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


feat_bench = assemble_features_bench(bench_queries, pools_bench, channels_bench)
print("feat_bench:", feat_bench.shape)

feat_bench: (882456, 31)


In [21]:
from catboost import CatBoostRanker

catboost_model = CatBoostRanker()
catboost_model.load_model("../data/catboost_reranker_prod.cbm")

feat_bench["score"] = catboost_model.predict(feat_bench[FEATURE_COLS])
feat_bench_sorted = feat_bench.sort_values(["query_id", "score"], ascending=[True, False])
final_predictions = feat_bench_sorted.groupby("query_id")["item_id"].apply(lambda s: list(s)[:50]).to_dict()

lens = [len(v) for v in final_predictions.values()]
print("запросов с ответом:", len(final_predictions), "| средняя длина ответа:", np.mean(lens), "| мин:", min(lens))

запросов с ответом: 2452 | средняя длина ответа: 50.0 | мин: 50


## answer.csv

In [22]:
missing = set(bench_queries["query_id"]) - set(final_predictions.keys())
print("запросов без кандидатов вообще:", len(missing))
for qid in missing:
    final_predictions[qid] = []

answer = pd.DataFrame({
    "query_id": bench_queries["query_id"],
    "answer": [" ".join(final_predictions.get(qid, [])) for qid in bench_queries["query_id"]],
})
assert len(answer) == len(bench_queries)
assert answer["query_id"].is_unique
assert (answer["answer"].str.split().str.len() <= 50).all()

SUBMISSION_PATH = "../submissions/v1.csv"
answer.to_csv(SUBMISSION_PATH, index=False)
print(f"{SUBMISSION_PATH} сохранён, строк:", len(answer))

запросов без кандидатов вообще: 0
answer.csv сохранён, строк: 2452
